<a href="https://colab.research.google.com/github/Wasey23/CI-345-Machine-Learning-Project/blob/main/CI345_Final_Project_Random_forest_%26_Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Data

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd # Added pandas import

movies = pd.read_csv('tmdb_movies_data.csv', engine='python', on_bad_lines='skip')
movies.head()


,id,imdb_id,popularity,budget,revenue,original_title,cast,homepage,director,tagline,...,overview,runtime,genres,production_companies,release_date,vote_count,vote_average,release_year,budget_adj,revenue_adj
0,135397,tt0369610,32.985763,150000000,1513528810,Jurassic World,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,http://www.jurassicworld.com/,Colin Trevorrow,The park is open.,...,Twenty-two years after the events of Jurassic ...,124,Action|Adventure|Science Fiction|Thriller,Universal Studios|Amblin Entertainment|Legenda...,6/9/2015,5562,6.5,2015,137999939.3,1.392446e+09
1,76341,tt1392190,28.419936,150000000,378436354,Mad Max: Fury Road,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,http://www.madmaxmovie.com/,George Miller,What a Lovely Day.,...,An apocalyptic story set in the furthest reach...,120,Action|Adventure|Science Fiction|Thriller,Village Roadshow Pictures|Kennedy Miller Produ...,5/13/2015,6185,7.1,2015,137999939.3,3.481613e+08
2,262500,tt2908446,13.112507,110000000,295238201,Insurgent,Shailene Woodley|Theo James|Kate Winslet|Ansel...,http://www.thedivergentseries.movie/#insurgent,Robert Schwentke,One Choice Can Destroy You,...,Beatrice Prior must confront her inner demons ...,119,Adventure|Science Fiction|Thriller,Summit Entertainment|Mandeville Films|Red Wago...,3/18/2015,2480,6.3,2015,101199955.5,2.716190e+08
3,140607,tt2488496,11.173104,200000000,2068178225,Star Wars: The Force Awakens,Harrison Ford|Mark Hamill|Carrie Fisher|Adam D...,http://www.starwars.com/films/star-wars-episod...,J.J. Abrams,Every generation has a story.,...,Thirty years after defeating the Galactic Empi...,136,Action|Adventure|Science Fiction|Fantasy,Lucasfilm|Truenorth Productions|Bad Robot,12/15/2015,5292,7.5,2015,183999919.0,1.902723e+09
4,168259,tt2820852,9.335014,190000000,1506249360,Furious 7,Vin Diesel|Paul Walker|Jason Statham|Michelle ...,http://www.furious7.com/,James Wan,Vengeance Hits Home,...,Deckard Shaw seeks revenge against Dominic Tor...,137,Action|Crime|Thriller,Universal Pictures|Original Film|Media Rights ...,4/1/2015,2947,7.3,2015,174799923.1,1.385749e+09


#Random forest

##Base features 'budget_adj', 'runtime' And 'release_month'

In [2]:
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# 1. Load the dataset and remove duplicates
df = movies.copy() # Changed to use the 'movies' DataFrame already loaded from Google Drive
df.drop_duplicates(inplace=True)

# 2. Define the target variable 'is_hit' (1 if revenue exceeded budget, else 0)
df["is_hit"] = (df["revenue_adj"] > df["budget_adj"]).astype(int)

# 3. Handle missing text fields
df["director"] = df["director"].fillna("")
df["cast"] = df["cast"].fillna("")

# 4. Feature Engineering: Extract top 20 most common directors
all_directors = []
for dir_list in df["director"].str.split("|"):
    all_directors.extend([d for d in dir_list if d != ""])
top_20_directors = [d[0] for d in Counter(all_directors).most_common(20)]

director_columns = []
for d in top_20_directors:
    clean_name = (
        d.replace(" ", "_")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
        .replace("'", "")
    )
    col_name = f"director_{clean_name}"
    df[col_name] = df["director"].apply(lambda x: 1 if d in x else 0)
    director_columns.append(col_name)

# 5. Feature Engineering: Extract top 20 most common cast members
all_cast = []
for cast_list in df["cast"].str.split("|"):
    all_cast.extend([c for c in cast_list if c != ""])
top_20_cast = [c[0] for c in Counter(all_cast).most_common(20)]

cast_columns = []
for c in top_20_cast:
    clean_name = (
        c.replace(" ", "_")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
        .replace("'", "")
    )
    col_name = f"cast_{clean_name}"
    df[col_name] = df["cast"].apply(lambda x: 1 if c in x else 0)
    cast_columns.append(col_name)

# 6. Select final features (Using budget_adj to handle inflation over time)
base_features = ["budget_adj"]
all_features = base_features + director_columns + cast_columns

X = df[all_features]
y = df["is_hit"]

print(f"Total feature columns being sent to Random Forest: {len(all_features)}")

# 7. Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8. Train the tuned Random Forest model matching your notebook configuration
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_split=5, random_state=42
)
rf_model.fit(X_train, y_train)

# 9. Make predictions and print results
y_pred = rf_model.predict(X_test)

print("\n=============================================")
print(f" MODEL OVERALL ACCURACY: {accuracy_score(y_test, y_pred):.2%}")
print("=============================================\n")

print("--- Detailed Performance Metrics ---")
print(
    classification_report(
        y_test, y_pred, target_names=["Not a Hit (0)", "Hit (1)"]
    )
)

Total feature columns being sent to Random Forest: 41

 MODEL OVERALL ACCURACY: 73.86%

--- Detailed Performance Metrics ---
               precision    recall  f1-score   support

Not a Hit (0)       0.80      0.81      0.80      1426
      Hit (1)       0.62      0.60      0.61       747

     accuracy                           0.74      2173
    macro avg       0.71      0.71      0.71      2173
 weighted avg       0.74      0.74      0.74      2173



##The Winning Combination: The most effective setup used Base Features + Studios + Genres + Keywords + Cast. This achieved an accuracy of 76.21%.

In [3]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Load data and clean
df = movies.copy() # Changed to use the 'movies' DataFrame already loaded from Google Drive
df.drop_duplicates(inplace=True)

# 2. Define the target 'is_hit'
df['is_hit'] = (df['revenue_adj'] > df['budget_adj']).astype(int)

# 3. Base Features
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_month'] = df['release_date'].dt.month.fillna(0).astype(int)
base_features = ['budget_adj', 'runtime', 'release_month']

# 4. Helper to encode multi-valued text columns (Top 20)
def encode_multi_valued(df, col, top_n=20):
    df[col] = df[col].fillna('')
    all_vals = []
    for val_list in df[col].str.split('|'):
        all_vals.extend([v for v in val_list if v != ''])
    top_vals = [v[0] for v in Counter(all_vals).most_common(top_n)]

    encoded_cols = []
    for v in top_vals:
        # Clean string to be a valid column name
        clean_name = v.replace(" ", "_").replace(".", "").replace("(", "").replace(")", "").replace("'", "").replace("-", "_")
        col_name = f"{col}_{clean_name}"
        df[col_name] = df[col].apply(lambda x: 1 if v in x else 0)
        encoded_cols.append(col_name)
    return encoded_cols

# 5. Build the Winning Feature Set
# Note: Excluding 'director' as per the winning configuration analysis
studio_cols = encode_multi_valued(df, 'production_companies', 20)
genre_cols = encode_multi_valued(df, 'genres', 20)
keyword_cols = encode_multi_valued(df, 'keywords', 20)
cast_cols = encode_multi_valued(df, 'cast', 20)

all_features = base_features + studio_cols + genre_cols + keyword_cols + cast_cols

# 6. Prepare X and y
X = df[all_features]
y = df['is_hit']

# 7. Train and Evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    random_state=42
)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print(f"Model Configuration: Base + Studios + Genres + Keywords + Cast")
print(f"Total Features Used: {len(all_features)}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\n--- Detailed Performance ---")
print(classification_report(y_test, y_pred, target_names=['Not a Hit (0)', 'Hit (1)']))

Model Configuration: Base + Studios + Genres + Keywords + Cast
Total Features Used: 83
Accuracy: 76.21%

--- Detailed Performance ---
               precision    recall  f1-score   support

Not a Hit (0)       0.78      0.88      0.83      1426
      Hit (1)       0.71      0.53      0.60       747

     accuracy                           0.76      2173
    macro avg       0.74      0.71      0.72      2173
 weighted avg       0.76      0.76      0.75      2173



#Naive Bayes models

##Base Beatures 'budget_adj', 'runtime' And 'release_month'

In [4]:
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# 1. Load the dataset and remove duplicates
df = movies.copy() # Changed to use the 'movies' DataFrame already loaded from Google Drive
df.drop_duplicates(inplace=True)

# 2. Define target variable (1 if revenue exceeded budget, else 0)
df["is_hit"] = (df["revenue_adj"] > df["budget_adj"]).astype(int)

# 3. Naive Bayes Transformation: Apply log transformation to the adjusted budget scale
df["log_budget"] = np.log1p(df["budget_adj"])

# 4. Handle missing text fields
df["director"] = df["director"].fillna("")
df["cast"] = df["cast"].fillna("")

# 5. Feature Engineering: Extract top 20 most common directors (1 or 0 columns)
all_directors = []
for dir_list in df["director"].str.split("|"):
    all_directors.extend([d for d in dir_list if d != ""])
top_20_directors = [d[0] for d in Counter(all_directors).most_common(20)]

director_columns = []
for d in top_20_directors:
    clean_name = (
        d.replace(" ", "_")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
        .replace("'", "")
    )
    col_name = f"director_{clean_name}"
    df[col_name] = df["director"].apply(lambda x: 1 if d in x else 0)
    director_columns.append(col_name)

# 6. Feature Engineering: Extract top 20 most common cast members (1 or 0 columns)
all_cast = []
for cast_list in df["cast"].str.split("|"):
    all_cast.extend([c for c in cast_list if c != ""])
top_20_cast = [c[0] for c in Counter(all_cast).most_common(20)]

cast_columns = []
for c in top_20_cast:
    clean_name = (
        c.replace(" ", "_")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
        .replace("'", "")
    )
    col_name = f"cast_{clean_name}"
    df[col_name] = df["cast"].apply(lambda x: 1 if c in x else 0)
    cast_columns.append(col_name)

# 7. Select features (Log budget + directors + cast)
base_features_nb = ["log_budget"]
all_features_nb = base_features_nb + director_columns + cast_columns

X = df[all_features_nb]
y = df["is_hit"]

print(f"Total features sent to Naive Bayes: {len(all_features_nb)}")

# 8. Split into Training (80%) and Testing (20%) sets using random_state=42 for consistency
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 9. Initialize and train the Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# 10. Make predictions and evaluate performance
y_pred_nb = nb_model.predict(X_test)

print("\n=============================================")
print(f" NAIVE BAYES MODEL OVERALL ACCURACY: {accuracy_score(y_test, y_pred_nb):.2%}")
print("=============================================\n")

print("--- Detailed Performance Metrics ---")
print(
    classification_report(
        y_test, y_pred_nb, target_names=["Not a Hit (0)", "Hit (1)"]
    )
)

Total features sent to Naive Bayes: 41

 NAIVE BAYES MODEL OVERALL ACCURACY: 67.37%

--- Detailed Performance Metrics ---
               precision    recall  f1-score   support

Not a Hit (0)       0.69      0.92      0.79      1426
      Hit (1)       0.57      0.20      0.29       747

     accuracy                           0.67      2173
    macro avg       0.63      0.56      0.54      2173
 weighted avg       0.65      0.67      0.62      2173



##The Winning Combination: Naive Bayes Model, the "All Combined (Except Director)" setup (Base + Studios + Genres + Keywords + Cast) yielded the highest accuracy at 71.19%.

In [5]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Load data and remove duplicates
df = movies.copy()
df.drop_duplicates(inplace=True)

# 2. Define the target variable 'is_hit'
df['is_hit'] = (df['revenue_adj'] > df['budget_adj']).astype(int)

# 3. Naive Bayes Preparation: Log transform budget to normalize the scale
df['log_budget'] = np.log1p(df['budget_adj'])

# 4. Feature Engineering: Basic numerical features
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_month'] = df['release_date'].dt.month.fillna(0).astype(int)
base_features_nb = ['log_budget', 'runtime', 'release_month']

# 5. Helper function to encode multi-valued columns (Top 20)
def encode_multi_valued(df, col, top_n=20):
    df[col] = df[col].fillna('')
    all_vals = []
    for val_list in df[col].str.split('|'):
        all_vals.extend([v for v in val_list if v != ''])
    top_vals = [v[0] for v in Counter(all_vals).most_common(top_n)]

    encoded_cols = []
    for v in top_vals:
        clean_name = v.replace(" ", "_").replace(".", "").replace("(", "").replace(")", "").replace("'", "").replace("-", "_")
        col_name = f"{col}_{clean_name}"
        df[col_name] = df[col].apply(lambda x: 1 if v in x else 0)
        encoded_cols.append(col_name)
    return encoded_cols

# 6. Apply encoding to categorical features (Excluding Director)
studio_cols = encode_multi_valued(df, 'production_companies', 20)
genre_cols = encode_multi_valued(df, 'genres', 20)
keyword_cols = encode_multi_valued(df, 'keywords', 20)
cast_cols = encode_multi_valued(df, 'cast', 20)

# Combine all features
all_features_nb = base_features_nb + studio_cols + genre_cols + keyword_cols + cast_cols

# 7. Prepare X and y
X = df[all_features_nb]
y = df['is_hit']

# 8. Split and Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# 9. Evaluate
y_pred_nb = nb_model.predict(X_test)

print(f"Model Configuration: Base + Studios + Genres + Keywords + Cast")
print(f"Total Features Used: {len(all_features_nb)}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.2%}")
print("\n--- Detailed Performance ---")
print(classification_report(y_test, y_pred_nb, target_names=['Not a Hit (0)', 'Hit (1)']))

Model Configuration: Base + Studios + Genres + Keywords + Cast
Total Features Used: 83
Accuracy: 71.19%

--- Detailed Performance ---
               precision    recall  f1-score   support

Not a Hit (0)       0.76      0.82      0.79      1426
      Hit (1)       0.59      0.51      0.55       747

     accuracy                           0.71      2173
    macro avg       0.68      0.66      0.67      2173
 weighted avg       0.70      0.71      0.71      2173



#Conclusion

In [6]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 1. Load data
df = movies.copy()
df.drop_duplicates(inplace=True)
df["is_hit"] = (df["revenue_adj"] > df["budget_adj"]).astype(int)
df["log_budget"] = np.log1p(df["budget_adj"])

# 2. Setup Data Prep Helper
def get_top_n_columns(df, col, n=20):
    df[col] = df[col].fillna("")
    all_vals = []
    for val_list in df[col].str.split("|"):
        all_vals.extend([v for v in val_list if v != ""])
    top_n = [v[0] for v in Counter(all_vals).most_common(n)]

    cols = []
    for v in top_n:
        clean_name = v.replace(" ", "_").replace(".", "").replace("(", "").replace(")", "").replace("'", "")
        col_name = f"{col}_{clean_name}"
        df[col_name] = df[col].apply(lambda x: 1 if v in x else 0)
        cols.append(col_name)
    return cols

# Prepare features
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_month'] = df['release_date'].dt.month.fillna(0).astype(int)

# Create expansion columns
studio_cols = get_top_n_columns(df, "production_companies")
genre_cols = get_top_n_columns(df, "genres")
keyword_cols = get_top_n_columns(df, "keywords")
cast_cols = get_top_n_columns(df, "cast")

# 3. Define Feature Sets
base_feats = ["budget_adj", "runtime", "release_month"]
expanded_feats = ["budget_adj", "runtime", "release_month"] + studio_cols + genre_cols + keyword_cols + cast_cols

base_feats_nb = ["log_budget", "runtime", "release_month"]
expanded_feats_nb = ["log_budget", "runtime", "release_month"] + studio_cols + genre_cols + keyword_cols + cast_cols

# 4. Evaluation Engine
results = []
y = df["is_hit"]

configs = [
    ("Random Forest: Base", RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42), base_feats),
    ("Random Forest: Expanded", RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42), expanded_feats),
    ("Naive Bayes  : Base", GaussianNB(), base_feats_nb),
    ("Naive Bayes  : Expanded", GaussianNB(), expanded_feats_nb)
]

for name, model, feats in configs:
    X_train, X_test, y_train, y_test = train_test_split(df[feats], y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    # Removed 'Total Features' from the dictionary below
    results.append({"Model Configuration": name, "Accuracy": f"{acc:.2%}"})

# 5. Display Final Leaderboard
master_table = pd.DataFrame(results)
print("\n==========================================================")
print("             MODEL PERFORMANCE LEADERBOARD                ")
print("==========================================================\n")
print(master_table.to_markdown(index=False))


             MODEL PERFORMANCE LEADERBOARD                

| Model Configuration     | Accuracy   |
|:------------------------|:-----------|
| Random Forest: Base     | 72.71%     |
| Random Forest: Expanded | 75.89%     |
| Naive Bayes  : Base     | 69.12%     |
| Naive Bayes  : Expanded | 71.19%     |
